## Imports

In [3]:
import os
from datetime import datetime, timedelta, timezone

import ee
import requests
import geopandas as gpd


In [4]:
ee.Authenticate()

True

In [5]:
ee.Initialize()

## Constants

In [6]:
CLOUD_FILTER = 60
CLD_PRB_THRESH = 50
NIR_DRK_THRESH = 0.15
CLD_PRJ_DIST = 1
BUFFER = 50

## Utils

In [ ]:
def _next_day(date_str: str) -> str:
    d = datetime.strptime(date_str, "%Y-%m-%d")
    return (d + timedelta(days=1)).strftime("%Y-%m-%d")


In [7]:
def get_s2_sr_cld_col(aoi_geom: ee.Geometry, start_date, end_date):
    """Retorna coleção S2 SR com s2cloudless joinado."""
    s2_sr_col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi_geom)
        .filterDate(start_date, end_date)
    )
    s2_cloudless_col = (
        ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
        .filterBounds(aoi_geom)
        .filterDate(start_date, end_date)
    )
    print(f"Total de imagens S2 SR:        {s2_sr_col.size().getInfo()}")
    print(f"Total de imagens s2cloudless:  {s2_cloudless_col.size().getInfo()}")

    return ee.ImageCollection(
        ee.Join.saveFirst("s2cloudless").apply(
            primary=s2_sr_col,
            secondary=s2_cloudless_col,
            condition=ee.Filter.equals(
                leftField="system:index", rightField="system:index"
            ),
        )
    )


In [36]:
def add_cloud_bands(img):
    cld_prb = ee.Image(img.get("s2cloudless")).select("probability")
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename("clouds")
    return img.addBands(ee.Image([cld_prb, is_cloud]))


In [37]:


def add_shadow_bands(img):
    not_water = img.select("SCL").neq(6)
    SR_BAND_SCALE = 1e4
    dark_pixels = (
        img.select("B8")
        .lt(NIR_DRK_THRESH * SR_BAND_SCALE)
        .multiply(not_water)
        .rename("dark_pixels")
    )
    shadow_azimuth = ee.Number(90).subtract(
        ee.Number(img.get("MEAN_SOLAR_AZIMUTH_ANGLE"))
    )
    cld_proj = (
        img.select("clouds")
        .directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST * 10)
        .reproject(crs=img.select(0).projection(), scale=10)
        .select("distance")
        .mask()
        .rename("cloud_transform")
    )
    shadows = cld_proj.multiply(dark_pixels).rename("shadows")
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))

In [38]:

def add_cld_shdw_mask(img):
    img_cloud = add_cloud_bands(img)
    img_cloud_shadow = add_shadow_bands(img_cloud)
    is_cld_shdw = (
        img_cloud_shadow.select("clouds")
        .add(img_cloud_shadow.select("shadows"))
        .gt(0)
    )
    is_cld_shdw = (
        is_cld_shdw.focalMin(2)
        .focalMax(BUFFER * 2 / 20)
        .reproject(crs=img.select([0]).projection(), scale=10)
        .rename("cloudmask")
    )
    return img_cloud_shadow.addBands(is_cld_shdw)


In [39]:
def _mosaic_by_date(
    collection: ee.ImageCollection,
    aoi_geom: ee.Geometry,
) -> list:
    """
    Agrupa imagens por data e faz mosaico recortado pela AOI.

    Retorna lista de dicts: [{"date": str, "image": ee.Image}, ...]

    Por que mosaico?
    ----------------
    O Sentinel-2 é distribuído por tiles de ~100 x 100 km. Se a AOI cruzar a
    fronteira entre dois tiles (ex.: 23KPQ e 23KPP), a mesma data terá duas
    imagens distintas. .mosaic() une os pixels das cenas do mesmo dia,
    eliminando lacunas na cobertura.
    """
    timestamps = collection.aggregate_array("system:time_start").getInfo()

    unique_dates = sorted(set(
        datetime.fromtimestamp(ts / 1000, tz=timezone.utc).strftime("%Y-%m-%d")
        for ts in timestamps
    ))

    mosaics = []
    for date_str in unique_dates:
        day_col = collection.filterDate(date_str, _next_day(date_str))
        n = day_col.size().getInfo()
        if n == 0:
            continue

        # mosaic() empilha as cenas (última data fica por cima);
        # clip() restringe o resultado à AOI — obrigatório antes do download.
        proj = day_col.first().select(0).projection()

        mosaic = (
            day_col
            .mosaic()
            .reproject(proj)
            # .clip(aoi_geom)
        )
        mosaics.append({"date": date_str, "image": mosaic})
        print(f"  [{date_str}] {n} cena(s) mosaicada(s)")

    return mosaics

## Downloads

In [61]:
def export_s2_cloud_shadow_masks(
    roi_feature_collection: ee.featurecollection.FeatureCollection,
    start_date,
    end_date,
    output_dir,
    scale: int = 10,
    cloud_filter: int = 60,
    cld_prb_thresh: int = 50,
    nir_drk_thresh: float = 0.15,
    cld_prj_dist: int = 1,
    buffer: int = 50,
    crs: str = "EPSG:4326",
):
    """
    Exporta localmente máscaras de nuvem/sombra Sentinel-2 via getDownloadURL().

    Parâmetros
    ----------
    roi_geometry    : ee.FeatureCollection | ee.Feature | ee.Geometry
    start_date      : str  – "YYYY-MM-DD"
    end_date        : str  – "YYYY-MM-DD"
    output_dir      : str  – pasta de saída (criada se não existir)
    scale           : int  – resolução em metros (padrão 10)
    crs             : str  – sistema de referência do GeoTIFF (padrão EPSG:4326)

    Bandas exportadas (GeoTIFF por data)
    ------------------------------------
    1. cloud_probability  – probabilidade s2cloudless (0-100)
    2. clouds             – pixels de nuvem (0/1)
    3. shadows            – pixels de sombra (0/1)
    4. cloud_shadow_mask  – máscara combinada e dilatada (0/1)
    """
    
    print(f"Parâmetros recebidos:")
    print(f"  ROI: {roi_feature_collection.getInfo()}")
    print(f"  Período: {start_date} a {end_date}")
    print(f"  Scale: {scale} m")
    # Atualiza globals de threshold
    global CLOUD_FILTER, CLD_PRB_THRESH, NIR_DRK_THRESH, CLD_PRJ_DIST, BUFFER
    CLOUD_FILTER    = cloud_filter
    CLD_PRB_THRESH  = cld_prb_thresh
    NIR_DRK_THRESH  = nir_drk_thresh
    CLD_PRJ_DIST    = cld_prj_dist
    BUFFER          = buffer
    
    # create all year in subfolder
    for year in range(int(start_date.split("-")[0]), int(end_date.split("-")[0]) + 1):
        year_dir = os.path.join(output_dir, str(year))
        os.makedirs(year_dir, exist_ok=True)

    os.makedirs(output_dir, exist_ok=True)

    collection = (
        get_s2_sr_cld_col(roi_feature_collection, start_date, end_date)
        .map(add_cld_shdw_mask)
    )

    n_total = collection.size().getInfo()
    print(f"\nTotal de imagens após processamento: {n_total}")
    if n_total == 0:
        print("Nenhuma imagem encontrada. Verifique AOI, datas e CLOUD_FILTER.")
        return

    # 4. Agrupa por data e mosaica
    print("\nAgrupando e mosaicando por data...")
    mosaics = _mosaic_by_date(collection, roi_feature_collection)
    print(f"\nTotal de datas únicas para exportar: {len(mosaics)}\n")

    # 5. Download por data
    for item in mosaics:
        date_str  = item["date"]
        mosaic_img = item["image"]
        export_img = (
            ee.Image.cat([
                mosaic_img.select("probability").rename("cloud_probability"),
                mosaic_img.select("clouds").rename("clouds"),
                mosaic_img.select("shadows").rename("shadows"),
                mosaic_img.select("cloudmask").rename("cloud_shadow_mask"),
            ])
            .toUint8()
        )

        roi_bounds = roi_feature_collection.geometry().bounds()
        print(roi_bounds.getInfo()["coordinates"])

        # export_img = export_img.clip(roi_bounds)

        filename = os.path.join(output_dir, date_str.split("-")[0] ,f"S2_cloud_shadow_mask_{date_str}.tif")

        # region recebe o dict GeoJSON — nunca o objeto ee.Geometry
        url = export_img.getDownloadURL({
            "scale":  scale,
            "region": roi_bounds,
            "format": "GEO_TIFF",
        })

        # print(f"Baixando {date_str} → {os.path.basename(filename)}")
        response = requests.get(url, stream=True)
        response.raise_for_status()

        with open(filename, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    print("\nDownload concluído.")

In [62]:
import geojson


def shapefile2feature_collection(
    shapefile: gpd.GeoDataFrame, *args, **kwargs
) -> ee.FeatureCollection:
    # Convert from GeoDataFrame to GeoJson
    geojson_data = geojson.loads(shapefile.to_json())

    # Create the FeatureCollection from geojson
    fc = ee.FeatureCollection(geojson_data)

    return fc

In [63]:

roi_feature_collection = shapefile2feature_collection(gpd.read_file('D:/GeoPipe/data/00_shapefiles/sume_reservatorio.geojson'))

In [64]:
print(type(roi_feature_collection))

<class 'ee.featurecollection.FeatureCollection'>


In [65]:
export_s2_cloud_shadow_masks(
    roi_feature_collection = roi_feature_collection,
    start_date = "2026-05-01",
    end_date = "2026-05-14",
    output_dir = "./",
    scale = 10,
)

Parâmetros recebidos:
  ROI: {'type': 'FeatureCollection', 'columns': {'system:index': 'String'}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-36.918867, -7.700604], [-36.917976, -7.699518], [-36.917474, -7.698816], [-36.916743, -7.699699], [-36.916058, -7.699586], [-36.916332, -7.698408], [-36.915075, -7.69868], [-36.915578, -7.697209], [-36.915327, -7.696575], [-36.915212, -7.695918], [-36.914641, -7.695896], [-36.914139, -7.696077], [-36.91327, -7.696145], [-36.912882, -7.695307], [-36.912357, -7.694854], [-36.912197, -7.695647], [-36.912334, -7.696416], [-36.911877, -7.696982], [-36.911786, -7.697775], [-36.911443, -7.697842], [-36.911786, -7.698454], [-36.911831, -7.69868], [-36.911831, -7.698884], [-36.911306, -7.699133], [-36.910529, -7.699291], [-36.910278, -7.699065], [-36.910324, -7.698771], [-36.90973, -7.69868], [-36.909364, -7.698431], [-36.909501, -7.698001], [-36.909935, -7.697865], [-36.909821, -7.697526], [-36.909752, -7.697095], 